# Built-in collections: lists, tuples, dicts, and sets

This module covers Python's four built-in collection types. Readers are assumed to know what an array, hash table, or named record is from another language, and to have at least passing familiarity with TM1 dimensions and cube cells; the focus here is on how Python realizes each collection, the methods that come up daily in tm1py work, and the trade-offs between the four.

The running examples draw on a small planning model: dimension element lists (`Period`, `Region`, `Product`, `Account`), cell coordinates as tuples, a cellset as `dict[tuple[str, ...], float]`, and sets used for element membership questions.

The topics below are arranged linearly for review. Structural grouping (sections, chapters) can be applied later.

---

## Topic list

1. The four built-in collections
2. Lists: creation
3. Lists: indexing and slicing
4. Lists: adding and removing
5. Lists: sorting and reversing
6. Lists: searching and counting
7. Lists: copying
8. Tuples: immutable sequences
9. Tuples: unpacking and packing
10. Named tuples
11. Dicts: creation and lookup
12. Dicts: adding, updating, removing
13. Dicts: iteration and views
14. Dicts: merging and combining
15. Specialized dicts
16. Sets: creation and membership
17. Set algebra
18. Frozen sets
19. Conversion between collections
20. Performance characteristics
21. Real-world design principles
22. Common mistakes

---

## 1. The four built-in collections

Python provides four general-purpose collections in the language core: `list`, `tuple`, `dict`, and `set`. Each one is the right answer to a different question.

| Type    | Ordered | Mutable | Indexed by | Duplicates | Typical use                          |
| ------- | ------- | ------- | ---------- | ---------- | ------------------------------------ |
| `list`  | yes     | yes     | integer    | yes        | sequence of items, in order          |
| `tuple` | yes     | no      | integer    | yes        | fixed record or coordinate           |
| `dict`  | yes     | yes     | hashable   | keys: no   | lookup table, named fields           |
| `set`   | no      | yes     | hashable   | no         | membership, deduplication, set logic |

"Ordered" for `dict` means insertion order is preserved (guaranteed since Python 3.7); it is not sorted order. "Indexed by hashable" means keys must support `hash()` — strings, numbers, tuples of hashables qualify; lists and dicts do not.

The four can be combined freely: a list of tuples, a dict whose values are sets, a set of frozen tuples. A typical tm1py shape — `dict[tuple[str, ...], float]` — is a dict keyed by coordinate tuples and valued by cell values.

In [ ]:
periods: list[str] = ["2026-Q1", "2026-Q2", "2026-Q3", "2026-Q4"]
regions: tuple[str, ...] = ("Europe", "North America", "Asia Pacific")
aliases: dict[str, str] = {"EUR": "Europe", "NA": "North America", "AP": "Asia Pacific"}
numeric_accounts: set[str] = {"Revenue", "COGS", "Operating Expense"}

The remaining topics walk through each type's creation, mutation, and query operations, and end with guidance on when to reach for which.

## 2. Lists: creation

Lists are written with square brackets. They can be empty, hold values of one type, or mix types (which is rarely a good idea but is permitted).

In [ ]:
empty: list[str] = []
periods: list[str] = ["2026-Q1", "2026-Q2", "2026-Q3", "2026-Q4"]
mixed: list = ["Revenue", 1500000.0, True]            # legal but discouraged

The `list()` constructor builds a list from any iterable. This is the standard way to convert a tuple, set, range, generator, or `dict_keys` view into a list.

In [ ]:
list(("a", "b", "c"))                  # ['a', 'b', 'c']
list(range(1, 5))                      # [1, 2, 3, 4]
list("Revenue")                        # ['R', 'e', 'v', 'e', 'n', 'u', 'e']
list({"Europe", "Asia"})               # order undefined: ['Europe', 'Asia'] or ['Asia', 'Europe']

A list of repeated values is constructed with `*`. The repeated value is shared, which matters when it is mutable (see Topic 22).

In [ ]:
zeros: list[float] = [0.0] * 12        # [0.0, 0.0, ..., 0.0] — 12 entries
flags: list[bool] = [False] * 4

Comprehensions are the most common way to build a list from another iterable; they are covered in `comprehensions.md`. The short version: `[expr for item in iterable if condition]` produces a new list.

In [ ]:
quarters: list[str] = [f"2026-Q{i}" for i in range(1, 5)]
# ['2026-Q1', '2026-Q2', '2026-Q3', '2026-Q4']

## 3. Lists: indexing and slicing

Indexes are zero-based. Negative indexes count from the end: `-1` is the last item, `-2` the second to last.

In [ ]:
periods = ["2026-Q1", "2026-Q2", "2026-Q3", "2026-Q4"]

periods[0]            # '2026-Q1'
periods[2]            # '2026-Q3'
periods[-1]           # '2026-Q4'
periods[-2]           # '2026-Q3'

periods[10]           # IndexError: list index out of range

Slicing returns a new list. The form is `list[start:stop:step]`. `start` is inclusive, `stop` is exclusive, `step` defaults to 1. Any of the three may be omitted.

In [ ]:
periods[1:3]          # ['2026-Q2', '2026-Q3']
periods[:2]           # ['2026-Q1', '2026-Q2']
periods[2:]           # ['2026-Q3', '2026-Q4']
periods[:]            # ['2026-Q1', '2026-Q2', '2026-Q3', '2026-Q4']  — full copy
periods[::2]          # ['2026-Q1', '2026-Q3']                         — every second
periods[::-1]         # ['2026-Q4', '2026-Q3', '2026-Q2', '2026-Q1']    — reversed

Out-of-range slicing does not raise; it is silently clamped. `periods[10:20]` returns `[]`. This is one of the few places in Python where invalid-looking input quietly produces a sensible result.

Slice assignment replaces a range of the list with another iterable, possibly of a different length:

In [ ]:
periods = ["2026-Q1", "2026-Q2", "2026-Q3", "2026-Q4"]
periods[1:3] = ["2026-H1"]            # replace two items with one
# ['2026-Q1', '2026-H1', '2026-Q4']

periods[1:1] = ["2026-Q2", "2026-Q3"]  # insert without removing

## 4. Lists: adding and removing

The most common mutations have dedicated methods. They modify the list in place and return `None`.

In [ ]:
elements: list[str] = []

elements.append("Revenue")              # add to the end
# ['Revenue']

elements.extend(["COGS", "Operating Expense"])   # add many to the end
# ['Revenue', 'COGS', 'Operating Expense']

elements.insert(1, "Gross Margin")      # insert at index, shifts later items right
# ['Revenue', 'Gross Margin', 'COGS', 'Operating Expense']

`+=` on a list calls `extend`, not `append`: `elements += ["X"]` adds one element, `elements += "X"` adds the characters `["X"]` (because a string is iterable). For a single-item add, `append` is clearer.

Removal happens by index, by value, or by slice.

In [ ]:
elements = ["Revenue", "Gross Margin", "COGS", "Operating Expense"]

elements.pop()                # 'Operating Expense' — removes and returns last
elements.pop(0)               # 'Revenue' — removes and returns at index
elements.remove("COGS")       # removes first matching value; ValueError if absent
del elements[0]               # removes at index, no return value
elements.clear()              # empties the list in place

`pop` is the right choice when the removed value is needed; `remove` is right when the value is known but its index is not; `del` is right when neither return nor lookup is needed.

## 5. Lists: sorting and reversing

Two forms: `list.sort()` modifies the list in place; `sorted(iterable)` returns a new list and accepts any iterable. Both take the same `key` and `reverse` arguments.

In [ ]:
periods = ["2026-Q3", "2026-Q1", "2026-Q4", "2026-Q2"]

periods.sort()                            # in place, returns None
# ['2026-Q1', '2026-Q2', '2026-Q3', '2026-Q4']

regions = ("Europe", "North America", "Asia Pacific")
sorted(regions)                           # new list from any iterable
# ['Asia Pacific', 'Europe', 'North America']

The `key` argument is a function (often a `lambda` or `operator.itemgetter`) that extracts the value to compare on. It is called once per element.

In [ ]:
elements = [
    ("Revenue", 1_500_000),
    ("COGS", 900_000),
    ("Operating Expense", 300_000),
]

sorted(elements, key=lambda pair: pair[1])
# [('Operating Expense', 300000), ('COGS', 900000), ('Revenue', 1500000)]

sorted(elements, key=lambda pair: pair[1], reverse=True)
# [('Revenue', 1500000), ('COGS', 900000), ('Operating Expense', 300000)]

from operator import itemgetter
sorted(elements, key=itemgetter(0))       # sort by first column (name)

Python's sort is stable: items that compare equal keep their relative order. This makes multi-key sorts easy by sorting in passes from least to most significant key, or by giving `key` a tuple.

In [ ]:
sorted(elements, key=lambda pair: (-pair[1], pair[0]))
# sort by amount descending, then name ascending

`list.reverse()` reverses in place; `reversed(iterable)` returns an iterator (not a list).

In [ ]:
periods.reverse()                         # in place
list(reversed(periods))                   # new list from iterator

## 6. Lists: searching and counting

The membership test is `in` and `not in`. It is O(n) for a list — every element may need to be examined. For frequent membership tests against a fixed collection, convert to a set first (Topic 16).

In [ ]:
elements = ["Revenue", "COGS", "Gross Margin"]

"Revenue" in elements           # True
"Profit" in elements            # False
"Profit" not in elements        # True

`list.index(value)` returns the position of the first match and raises `ValueError` if absent. Optional `start` and `stop` arguments restrict the search.

In [ ]:
elements.index("COGS")          # 1
elements.index("Profit")        # ValueError: 'Profit' is not in list
elements.index("Revenue", 1)    # ValueError — search starts at index 1

`list.count(value)` returns the number of occurrences. `0` if absent.

In [ ]:
amounts = [100.0, 200.0, 100.0, 50.0]
amounts.count(100.0)            # 2
amounts.count(999.0)            # 0

`min()`, `max()`, `sum()`, and the boolean reducers `any()` / `all()` work on lists directly:

In [ ]:
amounts = [100.0, 200.0, 100.0, 50.0]

min(amounts)                    # 50.0
max(amounts)                    # 200.0
sum(amounts)                    # 450.0
any(a > 150 for a in amounts)   # True
all(a > 0 for a in amounts)     # True

`min`/`max` accept `key` and `default`, just like `sorted`. `default` is returned when the iterable is empty (otherwise `ValueError`).

In [ ]:
max(amounts, default=0.0)
max((a for a in amounts if a > 1000), default=0.0)   # 0.0 — no match

## 7. Lists: copying

`elements_2 = elements_1` does not copy; it makes both names refer to the same list. Mutations through either name are visible through the other.

In [ ]:
a = ["Revenue", "COGS"]
b = a
b.append("Margin")
a                               # ['Revenue', 'COGS', 'Margin']  — a changed too

For an independent copy, use `list(a)`, `a[:]`, `a.copy()`, or `copy.copy(a)`. All produce a shallow copy: a new list containing the same element references.

In [ ]:
import copy

a = ["Revenue", "COGS"]
b = list(a)                     # or a[:], a.copy(), copy.copy(a)
b.append("Margin")
a                               # ['Revenue', 'COGS']  — unchanged
b                               # ['Revenue', 'COGS', 'Margin']

A shallow copy is sufficient when the elements are immutable (strings, numbers, tuples). For nested mutable structures, the inner objects are still shared.

In [ ]:
nested = [["Q1", 100.0], ["Q2", 200.0]]
shallow = nested[:]
shallow[0].append("revised")
nested                          # [['Q1', 100.0, 'revised'], ['Q2', 200.0]]  — inner list shared

deep = copy.deepcopy(nested)
deep[0].append("revised again")
nested                          # unchanged

`copy.deepcopy` is the right tool when independence at every level is required, but it is significantly slower than a shallow copy and follows references through arbitrary object graphs. Reach for it only when the shallow form is provably wrong.

## 8. Tuples: immutable sequences

A tuple is an ordered, indexable, immutable sequence. It supports indexing, slicing, iteration, and most non-mutating list methods, but `append`, `pop`, `sort`, and assignment to an index all raise `AttributeError` or `TypeError`.

In [ ]:
coord: tuple[str, str, str] = ("2026-Q1", "Europe", "Revenue")

coord[0]                        # '2026-Q1'
coord[-1]                       # 'Revenue'
coord[0:2]                      # ('2026-Q1', 'Europe')
len(coord)                      # 3
"Europe" in coord               # True

coord[0] = "2026-Q2"            # TypeError: 'tuple' object does not support item assignment

Tuples are written with parentheses, but the comma is what makes a tuple, not the parentheses. A single-element tuple needs a trailing comma; without it, the parentheses are just grouping.

In [ ]:
single: tuple[str] = ("Revenue",)        # tuple of length 1
not_a_tuple: str = ("Revenue")           # str — parentheses are redundant
empty: tuple = ()                        # empty tuple

Immutability applies to the tuple itself, not to its contents. A tuple of mutable objects can still have its contents mutated:

In [ ]:
weird: tuple[list[str], ...] = (["a"], ["b"])
weird[0].append("c")            # legal — the list inside is mutable
# (['a', 'c'], ['b'])
weird[0] = ["d"]                # TypeError — replacing the slot is not

Tuples of hashable elements are themselves hashable, which makes them usable as dict keys and set members. This is why TM1 cell coordinates are usually represented as tuples:

In [ ]:
cells: dict[tuple[str, str, str], float] = {
    ("2026-Q1", "Europe", "Revenue"): 1_500_000.0,
    ("2026-Q1", "Europe", "COGS"): 900_000.0,
    ("2026-Q1", "North America", "Revenue"): 2_100_000.0,
}

cells[("2026-Q1", "Europe", "Revenue")]   # 1500000.0

## 9. Tuples: unpacking and packing

Tuple unpacking assigns each element to a name. The number of names on the left must match the length of the tuple, unless `*` is used to absorb the rest.

In [ ]:
coord = ("2026-Q1", "Europe", "Revenue")
period, region, account = coord
period                          # '2026-Q1'
region                          # 'Europe'

coord = ("2026-Q1", "Europe", "Product A", "Revenue")
period, *middle, account = coord
middle                          # ['Europe', 'Product A']  — always a list

period, region, account = coord            # ValueError: too many values to unpack

The same syntax works for any iterable, not just tuples. It is the standard way to bind multiple return values from a function:

In [ ]:
def split_coord(coord: tuple[str, ...]) -> tuple[str, tuple[str, ...]]:
    return coord[0], coord[1:]

period, rest = split_coord(("2026-Q1", "Europe", "Revenue"))
# period = '2026-Q1', rest = ('Europe', 'Revenue')

Packing is the inverse: writing comma-separated values constructs a tuple. The parentheses are optional in most positions.

In [ ]:
def quarterly_totals() -> tuple[float, float, float, float]:
    return 100.0, 200.0, 150.0, 175.0     # tuple packing on return

q1, q2, q3, q4 = quarterly_totals()       # unpacking on assignment

The `*` operator on the call side unpacks an iterable into positional arguments; `**` unpacks a dict into keyword arguments:

In [ ]:
def cell(period: str, region: str, account: str) -> tuple[str, str, str]:
    return (period, region, account)

coord_args = ("2026-Q1", "Europe", "Revenue")
cell(*coord_args)                        # cell('2026-Q1', 'Europe', 'Revenue')

coord_kwargs = {"period": "2026-Q1", "region": "Europe", "account": "Revenue"}
cell(**coord_kwargs)

Swap is a one-liner thanks to packing and unpacking:

In [ ]:
a, b = 1, 2
a, b = b, a                              # a == 2, b == 1

## 10. Named tuples

A named tuple is a tuple subclass whose fields have names as well as positions. It keeps the lightweight semantics of a tuple (immutable, hashable, indexable, unpackable) while adding readable attribute access.

`typing.NamedTuple` is the modern way to define one; it accepts type hints and is recognized by static type checkers.

In [ ]:
from typing import NamedTuple


class CellCoord(NamedTuple):
    period: str
    region: str
    account: str


coord = CellCoord("2026-Q1", "Europe", "Revenue")

coord.period                    # '2026-Q1'
coord[0]                        # '2026-Q1' — still indexable
period, region, account = coord  # still unpacks
isinstance(coord, tuple)        # True

Named tuples are immutable. `_replace` returns a new instance with one or more fields replaced:

In [ ]:
adjusted = coord._replace(period="2026-Q2")
# CellCoord(period='2026-Q2', region='Europe', account='Revenue')

coord._asdict()
# {'period': '2026-Q1', 'region': 'Europe', 'account': 'Revenue'}

coord._fields
# ('period', 'region', 'account')

The older form `collections.namedtuple("CellCoord", ["period", "region", "account"])` produces the same kind of object but without type hints. New code should prefer `typing.NamedTuple`; the older form remains common in existing code.

When the values need to change, a `dataclass` (see `classes.md` Topic 22) is usually a better fit than reaching for `_replace` repeatedly.

## 11. Dicts: creation and lookup

A dict maps hashable keys to values. Literals use `{key: value, ...}`; the `dict()` constructor accepts keyword arguments or an iterable of pairs.

In [ ]:
empty: dict[str, str] = {}

aliases: dict[str, str] = {"EUR": "Europe", "NA": "North America", "AP": "Asia Pacific"}

regions = dict(EUR="Europe", NA="North America", AP="Asia Pacific")     # keyword form
pairs = dict([("EUR", "Europe"), ("NA", "North America")])               # iterable of pairs

Lookup with `d[key]` returns the value or raises `KeyError` if the key is absent. `d.get(key, default)` returns `default` (or `None`) instead of raising.

In [ ]:
aliases["EUR"]                  # 'Europe'
aliases["XX"]                   # KeyError: 'XX'

aliases.get("EUR")              # 'Europe'
aliases.get("XX")               # None
aliases.get("XX", "Unknown")    # 'Unknown'

`in` tests for key presence — never for value presence. To test a value, use `value in d.values()` (which is O(n)).

In [ ]:
"EUR" in aliases                # True — checks keys
"Europe" in aliases             # False — 'Europe' is a value, not a key
"Europe" in aliases.values()    # True

Dicts preserve insertion order. Iterating over a dict yields keys in the order they were first added.

In [ ]:
for code in aliases:
    print(code, aliases[code])
# EUR Europe
# NA North America
# AP Asia Pacific

## 12. Dicts: adding, updating, removing

Assignment with a new key adds an entry; assignment with an existing key replaces its value.

In [ ]:
aliases: dict[str, str] = {}
aliases["EUR"] = "Europe"
aliases["NA"] = "North America"
aliases["EUR"] = "Europe (EU + UK)"        # replaces existing value

`update` merges another mapping (or iterable of pairs) into the dict. Keys present in both are overwritten by the right-hand side.

In [ ]:
aliases.update({"AP": "Asia Pacific", "LATAM": "Latin America"})
aliases.update(MENA="Middle East & North Africa")          # keyword form

`setdefault(key, default)` returns the existing value if the key is present; otherwise it inserts the default and returns it. It is the canonical way to populate a dict whose values are themselves containers.

In [ ]:
by_region: dict[str, list[str]] = {}

for region, product in [
    ("Europe", "Laptop"),
    ("Europe", "Phone"),
    ("Asia Pacific", "Tablet"),
]:
    by_region.setdefault(region, []).append(product)

# {'Europe': ['Laptop', 'Phone'], 'Asia Pacific': ['Tablet']}

For deletion: `del d[key]` removes by key (KeyError if absent); `d.pop(key, default)` removes and returns; `d.popitem()` removes and returns the last inserted pair; `d.clear()` empties the dict.

In [ ]:
aliases = {"EUR": "Europe", "NA": "North America", "AP": "Asia Pacific"}

del aliases["AP"]                          # raises if absent
aliases.pop("NA")                          # 'North America'
aliases.pop("XX", None)                    # None — no exception
last_key, last_value = aliases.popitem()
aliases.clear()

## 13. Dicts: iteration and views

Three views: `d.keys()`, `d.values()`, `d.items()`. Each returns a view object — a lightweight, dynamic window onto the dict. Iterating a view does not copy. Changing the dict updates the view live.

In [ ]:
cells: dict[tuple[str, str], float] = {
    ("2026-Q1", "Europe"): 1_500_000.0,
    ("2026-Q1", "North America"): 2_100_000.0,
    ("2026-Q2", "Europe"): 1_650_000.0,
}

for coord, value in cells.items():
    print(coord, value)
# ('2026-Q1', 'Europe') 1500000.0
# ('2026-Q1', 'North America') 2100000.0
# ('2026-Q2', 'Europe') 1650000.0

list(cells.keys())            # [('2026-Q1', 'Europe'), ('2026-Q1', 'North America'), ('2026-Q2', 'Europe')]
list(cells.values())          # [1500000.0, 2100000.0, 1650000.0]
sum(cells.values())           # 5250000.0

Iterating directly over a dict (without `.items()`) yields keys. This is shorthand for `.keys()`.

In [ ]:
for coord in cells:
    ...                       # coord is the key

Sorting the iteration goes through `sorted()`:

In [ ]:
for coord, value in sorted(cells.items()):
    ...                                               # sorted by key
for coord, value in sorted(cells.items(), key=lambda kv: kv[1], reverse=True):
    ...                                               # sorted by value, descending

Key views support set-like operations directly, because keys are unique and hashable:

In [ ]:
a = {"Revenue": 1.0, "COGS": 2.0}
b = {"Revenue": 9.0, "OpEx": 4.0}

a.keys() & b.keys()           # {'Revenue'} — intersection
a.keys() | b.keys()           # {'Revenue', 'COGS', 'OpEx'} — union
a.keys() - b.keys()           # {'COGS'} — difference

## 14. Dicts: merging and combining

Three idioms produce a merged dict. `|` and `|=` (Python 3.9+) are the modern forms; `{**a, **b}` works in all current Pythons.

In [ ]:
defaults = {"currency": "EUR", "version": "Budget"}
overrides = {"version": "Forecast", "scenario": "Best"}

merged = defaults | overrides
# {'currency': 'EUR', 'version': 'Forecast', 'scenario': 'Best'}

defaults |= overrides            # in place; defaults now equals merged

merged_alt = {**defaults, **overrides}     # equivalent to | for simple dicts

When the same key appears in both, the right-hand side wins. This is consistent across all three forms and across `update`.

`update` is the one to reach for when the merge is one-way and in-place: `defaults.update(overrides)`. `|=` is equivalent and slightly more readable when the right-hand side is itself an expression.

For deep merges (combining nested dicts recursively), there is no built-in. Most projects either roll a small helper or use `mergedeep` from PyPI; the standard library does not solve this case.

## 15. Specialized dicts

The `collections` module provides three dict subclasses worth knowing.

`defaultdict` accepts a factory callable; missing-key access calls the factory and inserts the result. The result is the same as `setdefault` but written without the explicit insertion step.

In [ ]:
from collections import defaultdict

by_region: defaultdict[str, list[str]] = defaultdict(list)

for region, product in [("Europe", "Laptop"), ("Europe", "Phone"), ("Asia", "Tablet")]:
    by_region[region].append(product)

# defaultdict(<class 'list'>, {'Europe': ['Laptop', 'Phone'], 'Asia': ['Tablet']})

The factory can be `int` (zero counter), `set`, `list`, `dict`, or a custom callable. Once converted to a regular dict (e.g., for return), the default behavior is gone: `dict(by_region)`.

`Counter` is a dict subclass for counting hashables. It is the right tool for frequency tables, top-N queries, and multiset arithmetic.

In [ ]:
from collections import Counter

elements = ["Revenue", "COGS", "Revenue", "OpEx", "Revenue", "COGS"]
counts = Counter(elements)
# Counter({'Revenue': 3, 'COGS': 2, 'OpEx': 1})

counts.most_common(2)                # [('Revenue', 3), ('COGS', 2)]
counts["Revenue"]                    # 3
counts["Missing"]                    # 0 — never raises KeyError
counts.total()                       # 6 — sum of values

`OrderedDict` predates ordinary dict's order guarantee. The remaining reasons to reach for it are its `move_to_end` method and its order-sensitive equality (two `OrderedDict`s with the same items in different orders compare unequal, while two regular dicts compare equal). For new code, regular dicts are usually sufficient.

`ChainMap` views several dicts as one. Lookups walk through them in order. It is occasionally useful for layered configuration (defaults under environment overrides under runtime overrides) but is rarely the most readable choice.

## 16. Sets: creation and membership

A set is an unordered collection of unique hashable elements. Literals use braces; the `set()` constructor builds a set from any iterable.

In [ ]:
numeric_accounts: set[str] = {"Revenue", "COGS", "Operating Expense"}

empty: set[str] = set()                  # NOT {} — that is an empty dict

from_list = set(["Revenue", "COGS", "Revenue"])   # {'Revenue', 'COGS'}  — duplicates dropped
from_string = set("EUREKA")              # {'E', 'U', 'R', 'K', 'A'}  — characters

The empty set must be written `set()`. `{}` is an empty dict, not an empty set, because the dict literal got the syntax first.

Add and remove:

In [ ]:
accounts = {"Revenue", "COGS"}

accounts.add("OpEx")                     # idempotent — adding existing is a no-op
accounts.discard("OpEx")                 # remove if present, no error if absent
accounts.remove("Revenue")               # raises KeyError if absent
accounts.pop()                           # removes and returns an arbitrary element
accounts.clear()                         # empties in place

Membership tests are O(1) average case, the same as dict lookup. This is the main reason to use a set: when "is this in the collection?" is asked frequently against a sizeable collection, a set is dramatically faster than a list.

In [ ]:
elements = {"Revenue", "COGS", "Operating Expense", "Gross Margin", "Net Income"}

"Revenue" in elements                    # True — O(1)
"Profit" in elements                     # False

Sets are themselves not hashable (they are mutable), so a set of sets is not allowed. For that, see `frozenset` in Topic 18.

## 17. Set algebra

Sets support the full vocabulary of set operations as both operators and methods. Operators require both sides to be sets; methods accept any iterable.

In [ ]:
budget = {"Revenue", "COGS", "OpEx", "Tax"}
forecast = {"Revenue", "COGS", "OpEx", "FX Impact"}

budget | forecast                # union: in either set
# {'Revenue', 'COGS', 'OpEx', 'Tax', 'FX Impact'}

budget & forecast                # intersection: in both
# {'Revenue', 'COGS', 'OpEx'}

budget - forecast                # difference: in budget but not forecast
# {'Tax'}

budget ^ forecast                # symmetric difference: in either, but not both
# {'Tax', 'FX Impact'}

Each operator has a method form that accepts any iterable, not just a set:

In [ ]:
budget.union(["FX Impact", "Hedge"])
budget.intersection(["Revenue", "Margin"])
budget.difference(["Revenue"])
budget.symmetric_difference(["Tax", "Hedge"])

In-place variants modify the left operand: `|=`, `&=`, `-=`, `^=`. The named methods are `update`, `intersection_update`, `difference_update`, `symmetric_difference_update`.

In [ ]:
budget = {"Revenue", "COGS"}
budget |= {"OpEx"}                       # budget is now {'Revenue', 'COGS', 'OpEx'}

Subset and superset relationships have both operators and methods:

In [ ]:
core = {"Revenue", "COGS"}
extended = {"Revenue", "COGS", "OpEx"}

core <= extended                         # True — subset
core < extended                          # True — strict subset
extended >= core                         # True — superset
core.isdisjoint({"FX Impact"})           # True — no common elements

The set-algebra style is often the cleanest way to answer questions like "which elements are in production but missing from staging?" or "which dimensions are common between these two cubes?" without writing explicit loops.

## 18. Frozen sets

A `frozenset` is an immutable, hashable counterpart of `set`. It supports every non-mutating set method but no `add`, `discard`, `update`, or in-place operator.

In [ ]:
core: frozenset[str] = frozenset({"Revenue", "COGS"})

core | {"OpEx"}                          # works — returns a frozenset
core.add("OpEx")                         # AttributeError

Because frozensets are hashable, they can be members of other sets and keys of dicts:

In [ ]:
groups: dict[frozenset[str], str] = {
    frozenset({"Revenue", "COGS"}): "P&L core",
    frozenset({"Cash", "AR", "AP"}): "Working capital",
}

groups[frozenset({"COGS", "Revenue"})]   # 'P&L core' — order-insensitive lookup

The standard use case is a dict whose keys are unordered groups. Without `frozenset`, the same lookup would require sorting the input into a tuple first.

## 19. Conversion between collections

Each of the four constructors accepts any iterable, which means converting between them is direct:

In [ ]:
elements = ["Revenue", "COGS", "Revenue", "OpEx"]

list(elements)                  # ['Revenue', 'COGS', 'Revenue', 'OpEx']  — same content
tuple(elements)                 # ('Revenue', 'COGS', 'Revenue', 'OpEx')
set(elements)                   # {'Revenue', 'COGS', 'OpEx'}             — deduplicated
frozenset(elements)             # frozenset({'Revenue', 'COGS', 'OpEx'})

Converting from a dict iterates its keys by default. To convert items, ask explicitly:

In [ ]:
aliases = {"EUR": "Europe", "NA": "North America"}

list(aliases)                   # ['EUR', 'NA']                — keys only
list(aliases.items())           # [('EUR', 'Europe'), ('NA', 'North America')]
list(aliases.values())          # ['Europe', 'North America']

Converting to a dict requires pairs:

In [ ]:
pairs = [("EUR", "Europe"), ("NA", "North America")]
dict(pairs)                     # {'EUR': 'Europe', 'NA': 'North America'}

dict(zip(["EUR", "NA"], ["Europe", "North America"]))
# {'EUR': 'Europe', 'NA': 'North America'}

A common idiom for deduplicating while preserving order is `list(dict.fromkeys(items))`. It works because dicts preserve insertion order and only accept each key once.

In [ ]:
list(dict.fromkeys(["Revenue", "COGS", "Revenue", "OpEx"]))
# ['Revenue', 'COGS', 'OpEx']        — set() would lose the order

## 20. Performance characteristics

Average-case complexity for the operations that come up most often:

| Operation                  | list   | tuple  | dict   | set    |
| -------------------------- | ------ | ------ | ------ | ------ |
| `x in c`                   | O(n)   | O(n)   | O(1)   | O(1)   |
| `c[i]` by index            | O(1)   | O(1)   | n/a    | n/a    |
| `c[k]` by key              | n/a    | n/a    | O(1)   | n/a    |
| Append / add               | O(1)   | n/a    | O(1)   | O(1)   |
| Insert at position 0       | O(n)   | n/a    | n/a    | n/a    |
| Remove arbitrary value     | O(n)   | n/a    | O(1)   | O(1)   |
| Iterate                    | O(n)   | O(n)   | O(n)   | O(n)   |
| Sort                       | O(n log n) | n/a | n/a    | n/a    |

Two practical guidelines follow.

First, do not write `if x in long_list` inside a loop when the list does not change. Convert to a set once outside the loop and test membership against the set:

In [ ]:
# Slow: O(n*m)
for element in candidates:
    if element in known_elements_list:
        ...

# Fast: O(n + m)
known = set(known_elements_list)
for element in candidates:
    if element in known:
        ...

Second, avoid `list.insert(0, x)` and `del list[0]` for large lists. Use `collections.deque` if both ends of the sequence need cheap insertion and removal — it is O(1) at both ends.

## 21. Real-world design principles

Once the mechanics are clear, choosing among the four is the practical question.

**Use a list when order matters and access is by position.** Cell rows in display order, periods in chronological order, the result of a paginated read — anything where index matters and the collection grows or changes over time.

**Use a tuple when the collection is a fixed record or a coordinate.** A `(period, region, account)` cell key is a record of three positional fields, not a sequence of values that might grow. Tuples being hashable also makes them the natural key type for cellset-style dicts.

**Use a dict when access is by key.** Element name to element type, account code to account name, coordinate to value. The mental shorthand: if every "lookup" sentence in the design naturally says "given X, find Y," reach for a dict.

**Use a set when membership or set algebra is the operation.** "Which elements are missing from this dimension?", "Which accounts appear in both versions?", "Drop duplicates from this list." Sets are also the right answer when a list is being used purely for `in` tests on a stable population.

**Convert between them as needed.** Constructors are cheap; the right type for the operation is often not the right type for the storage. A common pattern: load a list of element names from tm1py, build a set for fast membership tests, but keep the original list around for ordered reporting.

**Prefer a named tuple or dataclass to a positional tuple of more than three fields.** A `(period, region, product, account, version)` raw tuple is hard to read at the call site and easy to misorder. `CellCoord(period=..., region=..., ...)` reads itself.

**Prefer the modern operators where they exist.** `dict_a | dict_b` for dict merge, `set_a | set_b` for set union, `|=` for in-place updates. They are clearer than the equivalent method calls and equally fast.

## 22. Common mistakes

A short collection of errors that are easy to make and worth recognizing early.

**Empty set vs. empty dict literal.** `{}` is an empty dict, not an empty set. To get an empty set, use `set()`.

In [ ]:
# Wrong
elements = {}
elements.add("Revenue")          # AttributeError: 'dict' object has no attribute 'add'

# Correct
elements = set()
elements.add("Revenue")

**Mutable default argument.** A list or dict used as a function default is shared across every call that does not pass an explicit value. The first call's mutations are visible to the second.

In [ ]:
# Wrong
def collect_elements(name: str, into: list[str] = []) -> list[str]:
    into.append(name)
    return into

collect_elements("Revenue")      # ['Revenue']
collect_elements("COGS")         # ['Revenue', 'COGS']  — same list!

# Correct
def collect_elements(name: str, into: list[str] | None = None) -> list[str]:
    if into is None:
        into = []
    into.append(name)
    return into

**Mutating a list while iterating it.** Adding or removing items from the same list a `for` loop is walking produces skipped or repeated elements. Iterate a copy, build a new list, or use a comprehension.

In [ ]:
# Wrong
elements = ["Revenue", "Revenue", "COGS", "Revenue"]
for e in elements:
    if e == "Revenue":
        elements.remove(e)       # skips items, leaves some "Revenue" entries

# Correct
elements = [e for e in elements if e != "Revenue"]
# or
for e in list(elements):         # iterate a snapshot
    if e == "Revenue":
        elements.remove(e)

**Repeating mutable values with `*`.** `[[]] * 3` produces three references to the same list, not three independent lists.

In [ ]:
# Wrong
grid = [[]] * 3
grid[0].append("a")
# [['a'], ['a'], ['a']]  — all share the same inner list

# Correct
grid = [[] for _ in range(3)]
grid[0].append("a")
# [['a'], [], []]

**Treating `dict` membership as a value test.** `value in d` checks keys, not values. To check values, ask `d.values()` explicitly.

In [ ]:
aliases = {"EUR": "Europe", "NA": "North America"}

# Wrong
if "Europe" in aliases:           # False — 'Europe' is a value, not a key
    ...

# Correct
if "Europe" in aliases.values():
    ...

**Using a list when a set was meant.** Repeated `in` tests against a long list are O(n) each time. Convert once to a set when the list does not change.

In [ ]:
known_elements = [...]            # a few thousand strings

# Wrong: O(n*m)
matches = [e for e in candidates if e in known_elements]

# Correct: build a set once, then test against it
known = set(known_elements)
matches = [e for e in candidates if e in known]

**Forgetting that dict keys must be hashable.** Trying to use a list (or another dict, or a set) as a key raises `TypeError`. Convert to a tuple (or `frozenset`) first.

In [ ]:
# Wrong
cells: dict = {}
cells[["2026-Q1", "Europe", "Revenue"]] = 1_500_000.0
# TypeError: unhashable type: 'list'

# Correct
cells[("2026-Q1", "Europe", "Revenue")] = 1_500_000.0

**Sorting by the wrong key shape.** `sorted(items, key=...)` calls the key function once per item. Returning the original item itself defeats the purpose; the key should be the value to sort by.

In [ ]:
elements = [("Revenue", 1_500_000), ("COGS", 900_000), ("OpEx", 300_000)]

# Wrong: sorts by the tuple, which compares lexicographically by name
sorted(elements)
# [('COGS', 900000), ('OpEx', 300000), ('Revenue', 1500000)]

# Correct: sort by amount
sorted(elements, key=lambda pair: pair[1])
# [('OpEx', 300000), ('COGS', 900000), ('Revenue', 1500000)]

**Assuming `sort()` returns the sorted list.** `list.sort()` mutates in place and returns `None`. `sorted()` returns a new list and accepts any iterable. Mixing them up produces `None` where a list was expected.

In [ ]:
elements = ["Revenue", "COGS", "OpEx"]

# Wrong
result = elements.sort()         # result is None
print(result[0])                 # TypeError: 'NoneType' object is not subscriptable

# Correct
elements.sort()                  # mutates in place
print(elements[0])
# or
result = sorted(elements)        # returns a new list
print(result[0])